# Notebook 04 – Visualization

K-core plot, ego-network, community colour map, degree×clustering scatter, summary grid.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import networkx as nx
from graph_utils import load_graph
from centrality import kcore_stats, top_degree
from community_detection import run_louvain

os.makedirs(os.path.join('..', 'figures'), exist_ok=True)

EDGE_FILE = os.path.join('..', 'data', 'com-dblp.ungraph.txt')
COMM_FILE = os.path.join('..', 'data', 'com-dblp.top5000.cmty.txt')
G, ground_truth = load_graph(EDGE_FILE, COMM_FILE)
print(f'Graph loaded: {G.number_of_nodes():,} nodes')

## 1. K-Core Subgraph Visualisation (high-core nodes)

In [ ]:
kc = kcore_stats(G)
max_core = kc['max_core']
threshold = max(max_core - 5, max_core // 2)

core_nodes = [n for n, c in kc['core_numbers'].items() if c >= threshold]
Gk = G.subgraph(core_nodes).copy()
print(f'Visualising {threshold}-core: {Gk.number_of_nodes()} nodes, {Gk.number_of_edges()} edges')

node_colors = [kc['core_numbers'][n] for n in Gk.nodes()]
pos = nx.spring_layout(Gk, seed=42, k=0.5)

fig, ax = plt.subplots(figsize=(10, 8))
nx.draw_networkx(
    Gk, pos=pos, ax=ax,
    node_color=node_colors, cmap='plasma',
    node_size=60, with_labels=False,
    edge_color='gray', alpha=0.7, width=0.5
)
sm = cm.ScalarMappable(cmap='plasma', norm=mcolors.Normalize(vmin=min(node_colors), vmax=max(node_colors)))
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Core number')
ax.set_title(f'K-Core Subgraph (k ≥ {threshold})')
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join('..', 'figures', 'kcore_subgraph.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Ego Network of Highest-Degree Node

In [ ]:
hub_node = top_degree(G, n=1)[0][0]
ego = nx.ego_graph(G, hub_node, radius=1)
print(f'Ego network of node {hub_node} (degree={G.degree(hub_node)}): {ego.number_of_nodes()} nodes')

pos_ego = nx.spring_layout(ego, seed=0)
node_sizes = [300 if n == hub_node else 30 for n in ego.nodes()]
node_cols  = ['red' if n == hub_node else 'steelblue' for n in ego.nodes()]

fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx(ego, pos=pos_ego, ax=ax,
                 node_size=node_sizes, node_color=node_cols,
                 with_labels=False, edge_color='gray', alpha=0.6, width=0.4)
ax.set_title(f'Ego Network – Hub Node {hub_node} (1-hop)')
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join('..', 'figures', 'ego_network.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Community Colour Map (sample of LCC)

In [ ]:
print('Running Louvain for colour map...')
partition = run_louvain(G, seed=42)

# Sample a manageable subgraph around high-degree nodes
sample_nodes = set()
for node, _ in top_degree(G, n=10):
    sample_nodes.update(nx.ego_graph(G, node, radius=1).nodes())

Gs = G.subgraph(sample_nodes).copy()
print(f'Sample subgraph: {Gs.number_of_nodes()} nodes')

unique_comms = sorted(set(partition[n] for n in Gs.nodes()))
cmap = plt.get_cmap('tab20', len(unique_comms))
comm_to_color = {c: cmap(i) for i, c in enumerate(unique_comms)}
node_cols = [comm_to_color[partition[n]] for n in Gs.nodes()]

pos_gs = nx.spring_layout(Gs, seed=1, k=0.6)

fig, ax = plt.subplots(figsize=(10, 8))
nx.draw_networkx(Gs, pos=pos_gs, ax=ax,
                 node_color=node_cols, node_size=40,
                 with_labels=False, edge_color='gray', alpha=0.5, width=0.3)
ax.set_title('Community Colour Map (hub neighbourhood sample)')
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join('..', 'figures', 'community_map.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Degree × Clustering Scatter

In [ ]:
# Sample to keep plot readable
import random
random.seed(42)
sample = random.sample(list(G.nodes()), min(5000, G.number_of_nodes()))

degs  = [G.degree(n) for n in sample]
clusts = list(nx.clustering(G, nodes=sample).values())

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(degs, clusts, alpha=0.3, s=8, c=degs, cmap='viridis')
plt.colorbar(sc, ax=ax, label='Degree')
ax.set_xlabel('Degree')
ax.set_ylabel('Clustering Coefficient')
ax.set_title('Degree vs. Clustering Coefficient (5,000-node sample)')
ax.set_xscale('log')
plt.tight_layout()
plt.savefig(os.path.join('..', 'figures', 'degree_clustering.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Summary Figure Grid

In [ ]:
from PIL import Image

fig_dir = os.path.join('..', 'figures')
panels = [
    ('degree_distribution.png', 'Degree Distribution'),
    ('kcore_subgraph.png',      'K-Core Subgraph'),
    ('community_map.png',       'Community Map'),
    ('degree_clustering.png',   'Degree × Clustering'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (fname, title) in zip(axes.flat, panels):
    path = os.path.join(fig_dir, fname)
    if os.path.exists(path):
        img = Image.open(path)
        ax.imshow(img)
    ax.set_title(title)
    ax.axis('off')

plt.suptitle('DBLP Network Analysis – Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'summary_grid.png'), dpi=150, bbox_inches='tight')
plt.show()